# DF-Arena Conformer LoRA 도메인 적응 (Colab)

**런타임 → GPU (A100 권장, L4/V100도 가능. T4는 batch=1·샘플 축소)**

준비물 (Google Drive):
1. 이 레포 코드 (`train_df_adapt.py`, `df_lora.py`, `learning_data.py`, …)
2. `model/df_arena_1b/` 전체 가중치
3. 오디오 + `data/manifests/train.csv`, `valid.csv` (경로가 Drive와 맞거나 아래 rewrite 사용)

산출물: `model/df_arena_lora.pt` → 로컬 `script.py` 제출 zip에 넣으면 VF에 자동 적용

## 0. GPU 확인

In [ ]:
!nvidia-smi
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 1. Drive 마운트 + 프로젝트 경로

`REPO`를 본인 Drive 경로로 바꾸세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ★ 수정: Drive에 올려 둔 레포 루트
REPO = "/content/drive/MyDrive/deepvoice_detecting"

import os
from pathlib import Path
os.chdir(REPO)
assert Path("train_df_adapt.py").is_file(), f"train_df_adapt.py 없음: {REPO}"
assert Path("model/df_arena_1b").is_dir(), "model/df_arena_1b 없음 — DF 가중치를 Drive에 올려야 함"
print("cwd:", Path.cwd())

## 2. 의존성

LoRA는 `peft` 없이 `df_lora.py`만 쓴다. DF 로드용으로 `transformers` 필요.

In [ ]:
!pip -q install -U librosa soxr soundfile tqdm transformers einops
# torch는 Colab 기본 CUDA 빌드 사용

## 3. 매니페스트 경로 확인

CSV 안 경로가 **로컬(D:\\…)** 이면 Colab에서 안 열립니다.

- **방법 A**: Drive에 오디오를 두고 `build_manifest`를 Drive 경로로 다시 생성
- **방법 B**: 아래 `REWRITE`로 접두사만 치환 (`--rewrite-prefix SRC=DST`)

In [ ]:
from pathlib import Path
import csv

train_csv = Path("data/manifests/train.csv")
valid_csv = Path("data/manifests/valid.csv")
print("train exists:", train_csv.is_file(), "valid exists:", valid_csv.is_file())

if train_csv.is_file():
    with train_csv.open(encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))[:5]
    for r in rows:
        print(r.get("source_folder"), "|", r.get("path", "")[:120])

# ★ 로컬→Drive 치환 예 (본인 환경에 맞게). 필요 없으면 빈 리스트.
# Windows 경로를 Drive로 옮긴 경우:
REWRITE = [
    # r"D:\Voice_Only_Zeroth=/content/drive/MyDrive/datasets/Voice_Only_Zeroth",
    # r"D:\=/content/drive/MyDrive/datasets/",
]

### (선택) Drive 경로로 매니페스트 새로 만들기

오디오가 Shareddrive/MyDrive에 폴더 규칙(`Voice_Only_*`, `Fake_Voice_Only_*`, …)으로 있으면:

In [ ]:
# 필요시에만 실행. DATA_ROOT를 본인 데이터 루트로.
RUN_BUILD_MANIFEST = False
DATA_ROOT = "/content/drive/MyDrive/datasets"  # ★

if RUN_BUILD_MANIFEST:
    !python -m preprocess.build_manifest --root "{DATA_ROOT}" --out data/manifests --valid-ratio 0.1
    print("manifest rebuilt")

## 4. LoRA 학습 실행

| GPU | 권장 |
|-----|------|
| A100 / L4 | `batch-size 2`, 아래 기본값 |
| T4 | `--batch-size 1 --max-per-source 80 --voice-per-source 300 --epochs 2` |

In [ ]:
rewrite_args = []
for item in REWRITE:
    rewrite_args += ["--rewrite-prefix", item]

cmd = [
    "python", "train_df_adapt.py",
    "--train-csv", "data/manifests/train.csv",
    "--valid-csv", "data/manifests/valid.csv",
    "--out", "model/df_arena_lora.pt",
    "--mode", "lora",
    "--last-n-blocks", "2",
    "--rank", "8",
    "--alpha", "16",
    "--epochs", "3",
    "--batch-size", "2",
    "--lr", "1e-4",
    "--max-per-source", "200",
    "--voice-per-source", "800",
    "--overlays", "200",
    "--phone-frac", "0.35",
    "--codec-frac", "0.25",
    "--band-frac", "0.2",
    "--domain-repeat", "2",
    "--device", "cuda",
] + rewrite_args

print(" ".join(cmd))
import subprocess, sys
proc = subprocess.run(cmd, cwd=str(Path.cwd()))
sys.exit(proc.returncode)

## 5. 결과 확인 → 로컬로 가져가기

1. Drive의 `model/df_arena_lora.pt` 다운로드
2. 로컬 레포 `model/df_arena_lora.pt`에 복사
3. 제출 zip에 **`df_arena_lora.pt` + `df_lora.py`** 포함
4. `script.py`는 파일만 있으면 자동 로드

```bash
python script.py --test-dir data/test --sample-submission data/sample_submission.csv --output output/submission.csv
```

In [ ]:
from pathlib import Path
p = Path("model/df_arena_lora.pt")
print("exists:", p.is_file(), "size_MB:", round(p.stat().st_size / 1e6, 2) if p.is_file() else None)
if p.is_file():
    import torch
    ckpt = torch.load(p, map_location="cpu", weights_only=False)
    print("meta:", ckpt.get("meta"))
    print("lora tensors:", len(ckpt.get("state_dict", {})))